In [ ]:
import os
from pathlib import Path

if not Path("/root/miniforge3").exists():
    !wget https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
    !bash Miniforge3-Linux-x86_64.sh -b -p /root/miniforge3
    !rm Miniforge3-Linux-x86_64.sh
else:
    print("Miniforge already installed.")

In [ ]:
!source /root/miniforge3/etc/profile.d/conda.sh && conda create -n lerobot_env python=3.12 -y
!source /root/miniforge3/etc/profile.d/conda.sh && conda activate lerobot_env && pip install --upgrade pip

In [ ]:
if not Path("/workspace/lerobot").exists():
    !git clone https://github.com/huggingface/lerobot.git /workspace/lerobot
else:
    print("LeRobot already cloned.")

In [ ]:
!source /root/miniforge3/etc/profile.d/conda.sh && conda activate lerobot_env && cd /workspace/lerobot && pip install --no-cache-dir -e ".[smolvla,dataset,viz]"

In [ ]:
!source /root/miniforge3/etc/profile.d/conda.sh && conda activate lerobot_env && python -c "import torch; print(f'PyTorch {torch.__version__} CUDA {torch.version.cuda}')"

In [ ]:
!source /root/miniforge3/etc/profile.d/conda.sh && conda activate lerobot_env && \
python -m lerobot.scripts.lerobot_train \
    --dataset.repo_id=leumals/ur5e_webots_pickplace \
    --policy.type=smolvla \
    --policy.pretrained_path=lerobot/smolvla_base \
    --output_dir=./outputs/train/smolvla_finetuned \
    --job_name=ur5e_pickplace_training \
    --batch_size=8 \
    --steps=20000 \
    --policy.device=cuda \
    --wandb.enable=false \
    --save_checkpoint=true \
    --policy.push_to_hub=false

In [ ]:
!source /root/miniforge3/etc/profile.d/conda.sh && conda activate lerobot_env && hf auth login

In [ ]:
!source /root/miniforge3/etc/profile.d/conda.sh && conda activate lerobot_env && \
hf upload leumals/smolvla_ur5e_finetuned \
    /workspace/lerobot/outputs/train/my_smolvla/checkpoints/last/pretrained_model .

In [ ]:
!source /root/miniforge3/etc/profile.d/conda.sh && conda activate lerobot_env && python -c "
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files('leumals/smolvla_ur5e_finetuned')
print('Uploaded files:', files)
"